In [1]:
import pandas as pd
import numpy as np

## Download ANES Data and Unpack

Download the ANES 2024 dataset from here: https://electionstudies.org/data-center/2024-time-series-study/, then unpack it and put the `anes_timeseries_2024_csv_20250808.csv` file into the data folder.

(I tried to automate this, but the ANES website was blocking `wget`)

## ANES Data Exploration

In [ ]:
df = pd.read_csv('data/anes_timeseries_2024_csv_20250808.csv', low_memory=False)
df

,version,V240001,V200001,V160001_orig,V240002a,V240002b,V240002c,V240003,V240101a,V240101b,...,V245009,V245010,V245011,V245012,V245013,V245014,V245015,V245016,V245017,V245018
0,ANES2024TimeSeries_20250808,140001,200015,401318,2,2,2,1,,,...,-1,-1. Inapplicable,-1,-1,-1,-1,-1,-1,-1,-1
1,ANES2024TimeSeries_20250808,140002,200022,300261,2,2,2,1,,,...,-1,-1. Inapplicable,-1,-1,-1,-1,-1,-1,-1,-1
2,ANES2024TimeSeries_20250808,140003,200039,400181,2,2,2,1,,,...,-1,-1. Inapplicable,-1,-1,-1,-1,-1,-1,-1,-1
3,ANES2024TimeSeries_20250808,140004,200046,300171,2,2,2,1,,,...,-1,-1. Inapplicable,-1,-1,-1,-1,-1,-1,-1,-1
4,ANES2024TimeSeries_20250808,140005,200053,405145,2,2,2,1,,,...,-1,-1. Inapplicable,-1,-1,-1,-1,-1,-1,-1,-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5516,ANES2024TimeSeries_20250808,399764,-1,-1,1,-6,1,3,.117236314673416,,...,0,-1. Inapplicable,-1,-1,-1,-1,-1,-1,-1,-1
5517,ANES2024TimeSeries_20250808,399815,-1,-1,1,5,2,3,1.50174751054799,1.15260127111406,...,1,IWER76,1960,3,0,1,5,0,1,0
5518,ANES2024TimeSeries_20250808,399830,-1,-1,1,-6,1,3,1.11447654600709,,...,2,-1. Inapplicable,-1,-1,-1,-1,-1,-1,-1,-1
5519,ANES2024TimeSeries_20250808,399841,-1,-1,4,4,2,3,1.17944749795356,1.1924221171493,...,2,IWER75,1950,3,0,1,5,0,1,2


Variables to note from the codebook below. I tried to match them as closely as possible to the 2016 / 2020 versions of the ANES that Argyle et al. (2023) used.

**Note that the patriotism (American flag) question was only included in the 2016 version of the ANES, but is missing in 2020 and 2024.**

- `V240107a` Pre-election full sample weight
- `V240107b` Post-election full sample weight
---
- `V241458x` PRE: SUMMARY: RESPONDENT AGE ON ELECTION
- `V242025` POST: HOW MANY DAYS IN PAST WEEK DISCUSSED POLITICS WITH FAMILY OR FRIENDS DAY
- `V241501x` PRE: SUMMARY: R SELF-IDENTIFIED RACE/ETHNICITY
- `V241177` PRE: 7PT SCALE LIBERAL-CONSERVATIVE SELF-PLACEMENT
- `V241707` SPS: R PARTY ID
- `V241439` PRE: EVER ATTEND CHURCH OR RELIGIOUS SERVICES
- `V241551` PRE: WHAT IS R’S GENDER?
- `V241023` PRE: REGISTRATION STATE (ALL REGISTRATIONS)
- `V243002` SAMPLE: SAMPLE LOCATION FIPS STATE CODE
---
- `V242095x` PRE-POST: SUMMARY: VOTER TURNOUT IN 2024
- `V242096x` PRE-POST: SUMMARY: 2024 PRESIDENTIAL VOTE

In [5]:
cols = {
  'V240107a': 'pre_weight',
  'V240107b': 'post_weight',
  'V241458x': 'age',
  'V242025': 'discuss_politics',
  'V241501x': 'race',
  'V241177': 'ideology',
  'V241707': 'party',
  'V241439': 'attend_church',
  'V241551': 'gender',
  'V242400': 'political_interest',
  'V241023': 'state',
  'V242095x': 'turnout',
  'V242096x': 'vote_choice'
}

short_df = df[cols.keys()]
short_df = short_df.rename(columns=cols)
short_df

,pre_weight,post_weight,age,discuss_politics,race,ideology,party,attend_church,gender,political_interest,state,turnout,vote_choice
0,0.716602,.710864874498513,50,2,3,6,-1,1,1,2,40,1,2
1,2.354582,2.38378558802356,41,1,4,4,-1,2,2,4,16,0,-2
2,0.780178,.808992408255851,44,7,1,2,-1,2,2,1,51,1,1
3,0.268988,.291739965421631,45,3,4,99,-1,1,1,3,6,0,-2
4,0.244306,.222012175174604,80,0,1,4,-1,2,1,3,8,1,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...
5516,0.193087,,-2,-6,-8,4,-1,1,2,-6,6,-2,-2
5517,2.910370,1.79786270502599,69,7,1,3,4,1,2,2,27,1,1
5518,2.112920,,-2,-6,2,99,-1,1,2,-6,51,1,-2
5519,2.635201,2.2559355310616,28,7,2,1,-1,2,1,1,34,1,1


In [6]:
cols.values()

dict_values(['pre_weight', 'post_weight', 'age', 'discuss_politics', 'race', 'ideology', 'party', 'attend_church', 'gender', 'political_interest', 'state', 'turnout', 'vote_choice'])

## ANES Preprocessing

In [ ]:
preproc_df = short_df.copy()

preproc_df['pre_weight'] = pd.to_numeric(preproc_df['pre_weight'], errors='coerce') # convert missing weights to np.nan
preproc_df['post_weight'] = pd.to_numeric(preproc_df['post_weight'], errors='coerce') # convert missing weights to np.nan

preproc_df['age'] = preproc_df.age.replace({-2: np.nan})
preproc_df['discuss_politics'] = preproc_df.discuss_politics.case_when([
    (preproc_df.discuss_politics > 0, True), # some time each week
    (preproc_df.discuss_politics == 0, False),
    ([True]*len(preproc_df), np.nan), # catch-all, missing values
])
preproc_df['race'] = preproc_df.race.case_when([
    (preproc_df.race == 1, 'White'), 
    (preproc_df.race == 2, 'Black'),
    (preproc_df.race == 3, 'Hispanic'),
    (preproc_df.race == 4, 'Asian'),
    (preproc_df.race == 5, 'Native American'),
    (preproc_df.race == 6, 'of multiple races'), # Template wording: "Racially, I am..."
    ([True]*len(preproc_df), np.nan), # catch-all, missing values
])
preproc_df['ideology'] = preproc_df.ideology.case_when([
    (preproc_df.ideology == 1, 'extremely liberal'),
    (preproc_df.ideology == 2, 'liberal'),
    (preproc_df.ideology == 3, 'slightly liberal'),
    (preproc_df.ideology == 4, 'moderate'),
    (preproc_df.ideology == 5, 'slightly conservative'),
    (preproc_df.ideology == 6, 'conservative'),
    (preproc_df.ideology == 7, 'extremely conservative'),
    (preproc_df.ideology == 99, np.nan), # "Haven’t thought much about this"
    ([True]*len(preproc_df), np.nan), # catch-all, missing values
])
preproc_df['party'] = preproc_df.party.case_when([
    (preproc_df.party == 1, 'a strong Democrat'),
    (preproc_df.party == 2, 'a weak Democrat'),
    (preproc_df.party == 3, 'an independent who leans Democratic'),
    (preproc_df.party == 4, 'an independent'),
    (preproc_df.party == 5, 'an independent who leans Republican'),
    (preproc_df.party == 6, 'a weak Republican'),
    (preproc_df.party == 7, 'a strong Republican'),
    ([True]*len(preproc_df), np.nan), # catch-all, missing values
])
preproc_df['attend_church'] = preproc_df.attend_church.case_when([
    (preproc_df.attend_church == 1, True),
    (preproc_df.attend_church == 2, False),
    ([True]*len(preproc_df), np.nan), # catch-all, missing values
])
preproc_df['gender'] = preproc_df.gender.case_when([
    (preproc_df.gender == 1, 'a man'),
    (preproc_df.gender == 2, 'a woman'),
    (preproc_df.gender == 3, 'nonbinary'),
    ([True]*len(preproc_df), np.nan), # catch-all, missing values, including "something else"
])
preproc_df['political_interest'] = preproc_df.political_interest.case_when([
    (preproc_df.political_interest == 1, 'very'),
    (preproc_df.political_interest == 2, 'somewhat'),
    (preproc_df.political_interest == 3, 'not very'),
    (preproc_df.political_interest == 3, 'not at all'),
    ([True]*len(preproc_df), np.nan), # catch-all, missing values
])
preproc_df['state'] = preproc_df.state.case_when([
    (preproc_df.state == 1, 'Alabama'),
    (preproc_df.state == 2, 'Alaska'),
    (preproc_df.state == 4, 'Arizona'),
    (preproc_df.state == 5, 'Arkansas'),
    (preproc_df.state == 6, 'California'),
    (preproc_df.state == 8, 'Colorado'),
    (preproc_df.state == 9, 'Connecticut'),
    (preproc_df.state == 10, 'Delaware'),
    (preproc_df.state == 11, 'Washington DC'),
    (preproc_df.state == 12, 'Florida'),
    (preproc_df.state == 13, 'Georgia'),
    (preproc_df.state == 15, 'Hawaii'),
    (preproc_df.state == 16, 'Idaho'),
    (preproc_df.state == 17, 'Illinois'),
    (preproc_df.state == 18, 'Indiana'),
    (preproc_df.state == 19, 'Iowa'),
    (preproc_df.state == 20, 'Kansas'),
    (preproc_df.state == 21, 'Kentucky'),
    (preproc_df.state == 22, 'Louisiana'),
    (preproc_df.state == 23, 'Maine'),
    (preproc_df.state == 24, 'Maryland'),
    (preproc_df.state == 25, 'Massachusetts'),
    (preproc_df.state == 26, 'Michigan'),
    (preproc_df.state == 27, 'Minnesota'),
    (preproc_df.state == 28, 'Mississippi'),
    (preproc_df.state == 29, 'Missouri'),
    (preproc_df.state == 30, 'Montana'),
    (preproc_df.state == 31, 'Nebraska'),
    (preproc_df.state == 32, 'Nevada'),
    (preproc_df.state == 33, 'New Hampshire'),
    (preproc_df.state == 34, 'New Jersey'),
    (preproc_df.state == 35, 'New Mexico'),
    (preproc_df.state == 36, 'New York'),
    (preproc_df.state == 37, 'North Carolina'),
    (preproc_df.state == 38, 'North Dakota'),
    (preproc_df.state == 39, 'Ohio'),
    (preproc_df.state == 40, 'Oklahoma'),
    (preproc_df.state == 41, 'Oregon'),
    (preproc_df.state == 42, 'Pennsylvania'),
    (preproc_df.state == 44, 'Rhode Island'),
    (preproc_df.state == 45, 'South Carolina'),
    (preproc_df.state == 46, 'South Dakota'),
    (preproc_df.state == 47, 'Tennessee'),
    (preproc_df.state == 48, 'Texas'),
    (preproc_df.state == 49, 'Utah'),
    (preproc_df.state == 50, 'Vermont'),
    (preproc_df.state == 51, 'Virginia'),
    (preproc_df.state == 53, 'Washington'),
    (preproc_df.state == 54, 'West Virginia'),
    (preproc_df.state == 55, 'Wisconsin'),
    (preproc_df.state == 56, 'Wyoming'),
    ([True]*len(preproc_df), np.nan), # catch-all, missing values
])
preproc_df['vote_choice'] = preproc_df.vote_choice.case_when([
    (preproc_df.vote_choice == 1, 'Harris'),
    (preproc_df.vote_choice == 2, 'Trump'),
    (preproc_df.turnout == 0, 'Non-voter'),
    # we might want to exclude the other candidates?
    #(preproc_df.vote_choice == 3, 'Kennedy'),
    #(preproc_df.vote_choice == 4, 'West'),
    #(preproc_df.vote_choice == 5, 'Stein'),
    #(preproc_df.vote_choice == 6, 'Other'),
    ([True]*len(preproc_df), np.nan), # catch-all, missing values
])
preproc_df

,pre_weight,post_weight,age,discuss_politics,race,ideology,party,attend_church,gender,political_interest,state,turnout,vote_choice
0,0.716602,0.710865,50.0,True,Hispanic,conservative,NaN,True,a man,somewhat,Oklahoma,1,Trump
1,2.354582,2.383786,41.0,True,Asian,moderate; middle of the road,NaN,False,a woman,NaN,Idaho,0,Non-voter
2,0.780178,0.808992,44.0,True,White,liberal,NaN,False,a woman,very,Virginia,1,Harris
3,0.268988,0.291740,45.0,True,Asian,NaN,NaN,True,a man,not very,California,0,Non-voter
4,0.244306,0.222012,80.0,False,White,moderate; middle of the road,NaN,False,a man,not very,Colorado,1,Trump
...,...,...,...,...,...,...,...,...,...,...,...,...,...
5516,0.193087,NaN,NaN,NaN,NaN,moderate; middle of the road,NaN,True,a woman,NaN,California,-2,NaN
5517,2.910370,1.797863,69.0,True,White,slightly liberal,an independent,True,a woman,somewhat,Minnesota,1,Harris
5518,2.112920,NaN,NaN,NaN,Black,NaN,NaN,True,a woman,NaN,Virginia,1,NaN
5519,2.635201,2.255936,28.0,True,Black,extremely liberal,NaN,False,a man,very,New Jersey,1,Harris


In [8]:
vote_reported_df = preproc_df.dropna(subset='vote_choice')
vote_reported_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4779 entries, 0 to 5520
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   pre_weight          4779 non-null   float64
 1   post_weight         4764 non-null   float64
 2   age                 4577 non-null   float64
 3   discuss_politics    4752 non-null   object 
 4   race                4737 non-null   object 
 5   ideology            4154 non-null   object 
 6   party               850 non-null    object 
 7   attend_church       4573 non-null   object 
 8   gender              4554 non-null   object 
 9   political_interest  4291 non-null   object 
 10  state               4270 non-null   object 
 11  turnout             4779 non-null   int64  
 12  vote_choice         4779 non-null   object 
dtypes: float64(3), int64(1), object(9)
memory usage: 522.7+ KB


In [9]:
vote_reported_df.to_csv('data/2024_anes_preprocessed.csv')